# Board Meeting RAG — 01: Ingest

Parses board-meeting documents from the raw Volume, chunks them with metadata
(entity, meeting_date, quarter, doc_type, page_number), and MERGEs into
`acme_holdings.documents.board_meetings_chunks`.

**Volume layout:** `/Volumes/acme_holdings/documents/board_meetings_raw/<quarter>/<file>`
where `<quarter>` is `YYYYQn` (e.g., `2026Q1`). Drop new docs under the right
quarter folder and re-run this notebook to refresh.


In [ ]:
%pip install --quiet pypdf python-docx
dbutils.library.restartPython()

In [ ]:
import hashlib
import re
from datetime import date, datetime
from pathlib import Path

from pypdf import PdfReader
from docx import Document as DocxDocument

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType, TimestampType,
)
from delta.tables import DeltaTable

CATALOG = "acme_holdings"
SCHEMA = "documents"
TABLE = "board_meetings_chunks"
FQN = f"{CATALOG}.{SCHEMA}.{TABLE}"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/board_meetings_raw"

# Per-entity metadata. Extend as you onboard more systems.
ENTITY_REGISTRY = {
    "SMIB": {
        "full_name": "State of Michigan Investment Board",
        "filename_match": re.compile(r"^SMIB", re.I),
    },
    "TCDRS": {
        "full_name": "Texas County & District Retirement System",
        "filename_match": re.compile(r"^TCDRS", re.I),
    },
    "SWIB": {
        "full_name": "State of Wisconsin Investment Board",
        "filename_match": re.compile(r"^SWIB", re.I),
    },
    "SDCERS": {
        "full_name": "San Diego City Employees' Retirement System",
        "filename_match": re.compile(r"^SDCERS", re.I),
    },
    "SFERS": {
        "full_name": "San Francisco City and County Employees' Retirement System",
        "filename_match": re.compile(r"^SFERS", re.I),
    },
    "CRPTF": {
        "full_name": "Connecticut Retirement Plans and Trust Funds",
        "filename_match": re.compile(r"^CRPTF", re.I),
    },
    "PASERS": {
        "full_name": "Pennsylvania State Employees' Retirement System",
        "filename_match": re.compile(r"^PASERS", re.I),
    },
    "VPIC": {
        "full_name": "Vermont Pension Investment Commission",
        "filename_match": re.compile(r"^VPIC", re.I),
    },
    "VRS": {
        "full_name": "Virginia Retirement System",
        "filename_match": re.compile(r"^VRS", re.I),
    },
    "MBOI": {
        "full_name": "Montana Board of Investments",
        "filename_match": re.compile(r"^MBOI", re.I),
    },
    "ILTRS": {
        "full_name": "Teachers' Retirement System of the State of Illinois",
        "filename_match": re.compile(r"^ILTRS", re.I),
    },
    "DCRB": {
        "full_name": "District of Columbia Retirement Board",
        "filename_match": re.compile(r"^DCRB", re.I),
    },
    "ACERA": {
        "full_name": "Alameda County Employees' Retirement Association",
        "filename_match": re.compile(r"^ACERA", re.I),
    },
    "ERSRI": {
        "full_name": "Employees' Retirement System of Rhode Island",
        "filename_match": re.compile(r"^ERSRI", re.I),
    },
    "KPERS": {
        "full_name": "Kansas Public Employees Retirement System",
        "filename_match": re.compile(r"^KPERS", re.I),
    },
    "ARMB": {
        "full_name": "Alaska Retirement Management Board",
        "filename_match": re.compile(r"^ARMB", re.I),
    },
    "SCERS": {
        "full_name": "Seattle City Employees' Retirement System",
        "filename_match": re.compile(r"^SCERS", re.I),
    },
    "NIC": {
        "full_name": "Nebraska Investment Council",
        "filename_match": re.compile(r"^NIC", re.I),
    },
    "OHSERS": {
        "full_name": "School Employees Retirement System of Ohio",
        "filename_match": re.compile(r"^OHSERS", re.I),
    },
    "TRSL": {
        "full_name": "Teachers' Retirement System of Louisiana",
        "filename_match": re.compile(r"^TRSL", re.I),
    },
    "LACERA": {
        "full_name": "Los Angeles County Employees Retirement Association",
        "filename_match": re.compile(r"^LACERA", re.I),
    },
}

MONTH_LOOKUP = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
}
MONTH_TO_QUARTER = {1: 1, 2: 1, 3: 1, 4: 2, 5: 2, 6: 2, 7: 3, 8: 3, 9: 3, 10: 4, 11: 4, 12: 4}

In [ ]:
def entity_from_filename(name: str) -> str:
    for code, meta in ENTITY_REGISTRY.items():
        if meta["filename_match"].search(name):
            return code
    return "UNKNOWN"

def quarter_from_path(path: str) -> str | None:
    m = re.search(r"/(\d{4}Q[1-4])/", path)
    return m.group(1) if m else None

def parse_month_year_heading(text: str) -> date | None:
    """Turn 'March 2026' into date(2026, 3, 1). Returns None if not a heading."""
    m = re.match(r"^\s*(january|february|march|april|may|june|july|august|september|october|november|december)\s+(\d{4})\s*$", text, re.I)
    if not m:
        return None
    return date(int(m.group(2)), MONTH_LOOKUP[m.group(1).lower()], 1)

def date_to_quarter(d: date) -> str:
    return f"{d.year}Q{MONTH_TO_QUARTER[d.month]}"

def chunk_id_for(source_file: str, page: int | None, ordinal: int, content: str) -> str:
    h = hashlib.sha1(content.encode("utf-8", errors="ignore")).hexdigest()[:10]
    return f"{Path(source_file).stem}:{page if page is not None else 'na'}:{ordinal}:{h}"

## PDF parsing — one chunk per page, preserves page_number

In [ ]:
def parse_pdf(local_path: str, source_file: str, entity: str, quarter: str) -> list[dict]:
    # Infer meeting_date from filename YYYY-MM-DD if present; else None
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})", source_file)
    meeting_date = date(int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else None

    reader = PdfReader(local_path)
    rows = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if not text:
            continue
        rows.append({
            "chunk_id": chunk_id_for(source_file, i, 0, text),
            "source_file": source_file,
            "entity": entity,
            "meeting_date": meeting_date,
            "quarter": quarter,
            "doc_type": "pdf",
            "page_number": i,
            "section": None,
            "content": text,
        })
    return rows

## DOCX parsing — split by `Month YYYY` headings, one chunk per meeting section

In [ ]:
def parse_docx(local_path: str, source_file: str, entity: str, fallback_quarter: str) -> list[dict]:
    doc = DocxDocument(local_path)
    # Collect non-empty paragraphs
    paras = [p.text.strip() for p in doc.paragraphs if p.text and p.text.strip()]

    # Walk paragraphs, split into sections at "Month YYYY" headings.
    sections: list[tuple[date | None, list[str]]] = []
    current_date: date | None = None
    current_buf: list[str] = []
    for text in paras:
        d = parse_month_year_heading(text)
        # Also handle "Past Board Meetings" as a section break without a date header
        if d is not None or text.lower() == "past board meetings":
            if current_buf:
                sections.append((current_date, current_buf))
            current_buf = []
            current_date = d if d is not None else current_date
            continue
        current_buf.append(text)
    if current_buf:
        sections.append((current_date, current_buf))

    rows = []
    for ordinal, (sec_date, buf) in enumerate(sections):
        body = "\n".join(buf).strip()
        if not body:
            continue
        sec_quarter = date_to_quarter(sec_date) if sec_date else fallback_quarter
        rows.append({
            "chunk_id": chunk_id_for(source_file, None, ordinal, body),
            "source_file": source_file,
            "entity": entity,
            "meeting_date": sec_date,
            "quarter": sec_quarter,
            "doc_type": "docx",
            "page_number": None,
            "section": sec_date.strftime("%B %Y") if sec_date else None,
            "content": body,
        })
    return rows

## Walk the Volume and collect all chunks

In [ ]:
def walk_volume(root: str):
    for entry in dbutils.fs.ls(root):
        if entry.isDir():
            yield from walk_volume(entry.path)
        else:
            yield entry.path  # dbfs:/Volumes/... style

all_rows: list[dict] = []
for dbfs_path in walk_volume(VOLUME_ROOT):
    # dbutils returns dbfs:/Volumes/... — strip scheme for local file reads
    local_path = dbfs_path.replace("dbfs:", "")
    filename = Path(local_path).name
    entity = entity_from_filename(filename)
    quarter = quarter_from_path(local_path) or "UNKNOWN"

    if filename.lower().endswith(".pdf"):
        all_rows.extend(parse_pdf(local_path, filename, entity, quarter))
    elif filename.lower().endswith(".docx"):
        all_rows.extend(parse_docx(local_path, filename, entity, quarter))
    else:
        print(f"Skipping unsupported file: {filename}")

print(f"Parsed {len(all_rows)} chunks from {VOLUME_ROOT}")

## MERGE into the chunks table (idempotent on chunk_id)

In [ ]:
schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("source_file", StringType(), False),
    StructField("entity", StringType(), True),
    StructField("meeting_date", DateType(), True),
    StructField("quarter", StringType(), True),
    StructField("doc_type", StringType(), True),
    StructField("page_number", IntegerType(), True),
    StructField("section", StringType(), True),
    StructField("content", StringType(), False),
])

staged = (
    spark.createDataFrame([Row(**r) for r in all_rows], schema=schema)
    .withColumn("ingested_at", F.current_timestamp())
)

target = DeltaTable.forName(spark, FQN)
(
    target.alias("t")
    .merge(staged.alias("s"), "t.chunk_id = s.chunk_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
display(
    spark.table(FQN)
    .groupBy("quarter", "entity", "doc_type")
    .agg(F.count("*").alias("chunks"), F.sum(F.length("content")).alias("chars"))
    .orderBy("quarter", "entity")
)